# SPOTIFY WRAPPED PROJECT (FIRST NOTEBOOK - DATA EXTRACTION)

## Table of Contents

1. **[SPOTIFY WRAPPED PROJECT – DATA EXTRACTION](#spotify-wrapped-project-first-notebook---data-extraction)**
   - [1.1 Purpose of This Notebook](#11-purpose-of-this-notebook)  
   - [1.2 How We Will Proceed](#12-how-we-will-proceed)  
   - [1.3 Python Libraries](#13-python-libraries)  
   - [1.4 Output of This Notebook](#14-output-of-this-notebook)  

2. **[Data Extraction](#2-data-extraction)**
   - [2.1 Environment & Setup](#21-environment--setup)  
   - [2.2 Load Environment Variables & Define Rutes](#22-load-environment-variables--define-rutes)  
   - [2.3 Create & Authenticate the Spotify Client](#23-create--authenticate-the-spotify-client)  
   - [2.4 Obtain and Save Top Tracks and Top Artists](#24-obtain-and-save-top-tracks-and-top-artists)  
         - [2.4.1 Helpers for Converting Data to DataFrames](#241-helpers-for-converting-data-to-dataframes)  
         - [2.4.2 Generate and Save the Most Popular Songs and Artists (short, medium and long term)](#242-generate-and-save-the-most-popular-songs-and-artists-short-medium-and-long-term)  
   - [2.5 Recently Played](#25-recently-played)  
   - [2.6 Extract Audio Features for All Your Songs](#26-extract-audio-features-for-all-your-songs)  
         - [2.6.1 Track IDs](#261-tracks-id)  

3. **[Summary](#3-summary)** 
   - [3.1 Note on Audio Features](#note-on-audio-features)  
   - [3.2 Output of This Notebook](#output-of-this-notebook)  


# 1. Introduction

This notebook is the starting point of the **Personal Spotify Wrapped** project.  
Its purpose is to connect to the Spotify Web API, authenticate using OAuth, and collect the raw data that will serve as the foundation for all subsequent analysis.

## 1.1 Purpose of This Notebook

The goals of this first step are:

1. **Authenticate with the Spotify Web API** using the Spotipy library.  
2. **Retrieve and store raw listening data**, including:
   - Recently played tracks  
   - Top artists  
   - Top tracks  
   - Audio features  
3. **Save all API responses** in the `data/raw/` directory in a reproducible format.  
4. Ensure that the structure and format of the saved data can be processed later in:
   - `02_eda.ipynb` (Exploratory Data Analysis)
   - `03_analysis.ipynb` (KPIs, storytelling, clustering, etc.)

This notebook does not clean or analyze data; its only responsibility is **data extraction and storage**.

## 1.2 How We Will Proceed

1. **Load environment variables**  
   We will load the Spotify API credentials from a `.env` file (client ID, client secret, redirect URI).

2. **Authenticate with Spotify**  
   Using Spotipy’s `SpotifyOAuth`, we will request the required scopes and generate a token.

3. **Fetch raw data from selected Spotify API endpoints**  
   Depending on scope availability:
   - User top tracks  
   - User top artists  
   - Recently played items  
   - Audio features for tracks  

4. **Normalize JSON responses**  
   Convert Spotify’s nested JSON outputs into structured tabular format (pandas DataFrames).

5. **Save the raw outputs**  
   All unmodified data will be stored in:
       `data/raw/`

## 1.3 Python libraries:
  - spotipy
  - python-dotenv
  - pandas
  - numpy

## 1.4 Output of This Notebook

At the end of this notebook, you will have a complete set of raw Spotify data stored locally.  
This ensures reproducibility and a clear separation between data extraction, data cleaning, and analysis.

We can now proceed with the implementation.


# 2. Data Extraction

## 2.1 Environment & Setup

In [1]:
# Standard library imports 
import os
from pathlib import Path

# Third-party imports
import spotipy
from spotipy.oauth2 import SpotifyOAuth
from dotenv import load_dotenv
import pandas as pd


## 2.2 Load Environment Variables & Define Rutes

In [ ]:
# Load environment variables from .env file 
load_dotenv()

CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
REDIRECT_URI = os.getenv("SPOTIFY_REDIRECT_URI")

print("Client ID loaded:", CLIENT_ID[:5] + "*****" if CLIENT_ID else "NOT FOUND")

# Detect project root from notebook location
current_path = Path().resolve()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", DATA_RAW_DIR)


Client ID loaded: 2ce1f*****
Project root: C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project
Raw data directory: C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project\data\raw


## 2.3 Create & Authenticate the Spotify Client

In [4]:
# Create Spotify client using OAuth
def create_spotify_client(
    scope: str = "user-read-recently-played user-top-read"
) -> spotipy.Spotify:
    """
    Create an authenticated Spotify client.
    """
    if CLIENT_ID is None or CLIENT_SECRET is None or REDIRECT_URI is None:
        raise ValueError("Spotify credentials not found. Check your .env file.")
    
    auth_manager = SpotifyOAuth(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        redirect_uri=REDIRECT_URI,
        scope=scope,
        show_dialog=True
    )
    
    sp = spotipy.Spotify(auth_manager=auth_manager)
    return sp

sp = create_spotify_client()

# Test authentication
current_user = sp.current_user()
print("Authenticated as:", current_user["display_name"])

Authenticated as: Sergioosnchz


## 2.4 Obtain and Save Top Tracks and Top Artists

### 2.4.1 Helpers for Converting Data to DataFrames

In [5]:
def fetch_top_items(sp_client, item_type="tracks", time_range="short_term", limit=50):
    """
    Fetch top tracks or artists for the current user.
    """
    if item_type == "tracks":
        results = sp_client.current_user_top_tracks(time_range=time_range, limit=limit)
    elif item_type == "artists":
        results = sp_client.current_user_top_artists(time_range=time_range, limit=limit)
    else:
        raise ValueError("item_type must be 'tracks' or 'artists'.")
    return results["items"]


def top_tracks_to_df(items):
    """
    Convert Spotify top tracks response into a DataFrame.
    """
    rows = []
    for idx, item in enumerate(items, start=1):
        rows.append({
            "rank": idx,
            "track_id": item["id"],
            "track_name": item["name"],
            "artist_names": ", ".join(a["name"] for a in item["artists"]),
            "album_name": item["album"]["name"],
            "duration_ms": item["duration_ms"],
            "explicit": item["explicit"],
            "popularity": item["popularity"]
        })
    return pd.DataFrame(rows)


def top_artists_to_df(items):
    """
    Convert Spotify top artists response into a DataFrame.
    """
    rows = []
    for idx, item in enumerate(items, start=1):
        rows.append({
            "rank": idx,
            "artist_id": item["id"],
            "artist_name": item["name"],
            "genres": ", ".join(item.get("genres", [])),
            "popularity": item["popularity"],
            "followers": item["followers"]["total"]
        })
    return pd.DataFrame(rows)

### 2.4.2 Generate and Save the Most Popular Songs and Artists (short, medium and long term)

In [6]:
time_ranges = ["short_term", "medium_term", "long_term"]

for tr in time_ranges:
    # Top tracks
    top_tracks = fetch_top_items(sp, item_type="tracks", time_range=tr, limit=50)
    df_tracks = top_tracks_to_df(top_tracks)
    path_tracks = DATA_RAW_DIR / f"top_tracks_{tr}.csv"
    df_tracks.to_csv(path_tracks, index=False)
    print(f"Saved top tracks ({tr}) -> {path_tracks}")
    
    # Top artists
    top_artists = fetch_top_items(sp, item_type="artists", time_range=tr, limit=50)
    df_artists = top_artists_to_df(top_artists)
    path_artists = DATA_RAW_DIR / f"top_artists_{tr}.csv"
    df_artists.to_csv(path_artists, index=False)
    print(f"Saved top artists ({tr}) -> {path_artists}")


Saved top tracks (short_term) -> C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project\data\raw\top_tracks_short_term.csv
Saved top artists (short_term) -> C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project\data\raw\top_artists_short_term.csv
Saved top tracks (medium_term) -> C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project\data\raw\top_tracks_medium_term.csv
Saved top artists (medium_term) -> C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project\data\raw\top_artists_medium_term.csv
Saved top tracks (long_term) -> C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project\data\raw\top_tracks_long_term.csv
Saved top artists (long_term) -> C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped

## 2.5 Recently Played

In [7]:
def fetch_recently_played(sp_client, limit: int = 50) -> pd.DataFrame:
    """
    Fetch recently played tracks for the current user.

    Parameters
    ----------
    sp_client : spotipy.Spotify
        Authenticated Spotify client.
    limit : int
        Number of items to fetch (max 50 by Spotify API).

    Returns
    -------
    pandas.DataFrame
        DataFrame with recently played tracks and timestamps.
    """
    results = sp_client.current_user_recently_played(limit=limit)
    items = results.get("items", [])

    rows = []
    for item in items:
        track = item["track"]
        rows.append({
            "played_at": item["played_at"],
            "track_id": track["id"],
            "track_name": track["name"],
            "artist_names": ", ".join(a["name"] for a in track["artists"]),
            "album_name": track["album"]["name"],
            "duration_ms": track["duration_ms"],
            "explicit": track["explicit"],
            "popularity": track["popularity"]
        })

    return pd.DataFrame(rows)


df_recent = fetch_recently_played(sp, limit=50)

recent_path = DATA_RAW_DIR / "recently_played.csv"
df_recent.to_csv(recent_path, index=False)

print("Saved recently played tracks to:", recent_path)
df_recent.head()


Saved recently played tracks to: C:\Users\Sergio\Desktop\SERGIO\DATA SCIENCE\GITHUB INGLES\data-science-portfolio\spotify-wrapped-project\data\raw\recently_played.csv


,played_at,track_id,track_name,artist_names,album_name,duration_ms,explicit,popularity
0,2025-12-01T22:06:26.175Z,4fZj0uoOsTwAgSyHLL9xbV,0640,Kidd Keo,Old Times in Gotham,112725,True,35
1,2025-12-01T22:04:33.184Z,79X7BhWIBGSjChXPjkJcvQ,HOPE (with Kidd Keo),"TALE$, Kidd Keo",HOPE (with Kidd Keo),159076,False,54
2,2025-12-01T22:01:53.363Z,2wIoxC17DLeB3zinoQykmX,Stress,Kidd Keo,Stress,187951,True,38
3,2025-12-01T21:58:40.674Z,4jaNOoKNuh8UejoZQRm0OZ,Triple Six,Hoke,TRES CREUS,126956,True,51
4,2025-12-01T21:14:06.248Z,1gcTe8hDGX9FNLUHAP4XQG,Nos Creíamos Kies,"Hoke, Morad",TRES CREUS,212389,True,49


## 2.6 Extract Audio Features for All Your Songs

### 2.6.1 Track's ID

In [8]:
def load_all_top_tracks_ids(raw_dir: Path) -> list:
    """
    Load all 'top_tracks_*.csv' files from raw_dir and return unique track IDs.
    """
    track_ids = set()
    for csv_file in raw_dir.glob("top_tracks_*.csv"):
        df = pd.read_csv(csv_file)
        track_ids.update(df["track_id"].dropna().tolist())
    return list(track_ids)


top_track_ids = load_all_top_tracks_ids(DATA_RAW_DIR)

# From recently played
recent_track_ids = df_recent["track_id"].dropna().tolist()

# Union of both
all_track_ids = list(set(top_track_ids + recent_track_ids))

print("Top tracks IDs:", len(top_track_ids))
print("Recent track IDs:", len(recent_track_ids))
print("Unique track IDs total:", len(all_track_ids))


Top tracks IDs: 95
Recent track IDs: 50
Unique track IDs total: 121


## 3. Summary

This notebook completes the first stage of the **Personal Spotify Wrapped** project: collecting all the raw data required for the later analysis and storytelling components.

The main objectives of this notebook were:

1. **Authenticate with the Spotify Web API** using OAuth through the Spotipy client.
2. **Extract listening-related datasets**, including:
   - Top Tracks (short-term, medium-term, long-term)
   - Top Artists (short-term, medium-term, long-term)
   - Recently Played tracks
3. **Store all results in the `data/raw/` directory** in a clean and reproducible file structure.

These raw datasets serve as the foundation for the next notebook, where the Exploratory Data Analysis (EDA) will be performed.

### 3.1 Note on Audio Features

The original plan included extracting track-level audio features (such as energy, danceability, or valence) via Spotify’s `audio-features` endpoint.  
However, this endpoint is currently restricted for many newly created applications and returns `403 Forbidden` errors even with proper authentication.

To maintain a fully reproducible and functional workflow, this step has been **intentionally omitted** in the project.  
The rest of the analysis—including listening habits, rankings, temporal patterns, KPIs, and storytelling—remains fully supported by the available datasets.

### 3.2 Output of This Notebook

By the end of this notebook, the following raw data files have been generated:

- `top_tracks_short_term.csv`
- `top_tracks_medium_term.csv`
- `top_tracks_long_term.csv`
- `top_artists_short_term.csv`
- `top_artists_medium_term.csv`
- `top_artists_long_term.csv`
- `recently_played.csv`

These files will be loaded and explored in the next notebook:  
**`02_eda.ipynb` — Exploratory Data Analysis**.
